# MC Dropout Uncertainty Quantification

## CBIS-DDSM Mammography Classification

This notebook implements Monte Carlo Dropout for uncertainty quantification in breast cancer classification. The model provides calibrated probability estimates with predictive variance to identify cases requiring clinical review.

### Configuration
- Architecture: DenseNet-121 with 2048 dense units
- MC Samples: 75 forward passes
- Calibration: Ensemble selection (Temperature, Platt, Isotonic)

### Targets
| Metric | Target | Description |
|--------|--------|-------------|
| ECE | < 0.10 | Expected Calibration Error |
| Variance | >= 0.01 | Mean predictive variance |
| Ratio | >= 1.5x | Incorrect/correct variance ratio |

## 1. Setup
Mount Google Drive (Colab) or set local paths. Add notebook directory to Python path.

In [ ]:
# Setup
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
 from google.colab import drive
 drive.mount('/content/drive')
 BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 NOTEBOOK_PATH = Path('/content/drive/MyDrive/Colab Notebooks')
 sys.path.insert(0, str(NOTEBOOK_PATH))
else:
 BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 NOTEBOOK_PATH = BASE_PATH
 sys.path.insert(0, str(NOTEBOOK_PATH))

## 2. Imports
Load TensorFlow, NumPy, and the mc_dropout package for uncertainty quantification.

In [ ]:
# Imports
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

from mc_dropout import (
 UncertaintyConfig,
 MCDropoutModel,
 ClinicalDecisionMaker,
 ConfidenceLevel,
 TemperatureScaling,
 PlattScaling,
 IsotonicCalibration
)

## 3. Configuration
Define MC Dropout hyperparameters: 75 samples, 65% MC dropout, 2048 dense units.

In [ ]:
# Configuration
config = UncertaintyConfig(
 n_mc_samples=75,
 batch_size=32,
 dense_units=2048,
 dropout_rate=0.35,
 mc_dropout_rate=0.65,
 spatial_dropout_rate=0.35,
 use_spatial_dropout=True,
 high_confidence_threshold=0.85,
 low_confidence_threshold=0.50,
)

## 4. Load Data
Load preprocessed ROI image caches for test and validation sets.

In [ ]:
# Load Data
cache_path = BASE_PATH / 'preprocessed_cache_roi'

test_cache = np.load(cache_path / 'test_cache.npz')
test_images = test_cache['images']
test_labels = test_cache['labels']

val_cache = np.load(cache_path / 'val_cache.npz')
val_images = val_cache['images']
val_labels = val_cache['labels']

print(f"Test set: {len(test_images)} samples ({test_labels.mean():.1%} positive)")
print(f"Validation set: {len(val_images)} samples ({val_labels.mean():.1%} positive)")

## 5. Build Model
Create MC Dropout model and load trained weights from checkpoint.

In [ ]:
# Build and Load Model
mc_model = MCDropoutModel(config)

checkpoint_path = BASE_PATH / 'checkpoints_roi' / 'model_0_stage3_best.h5'
mc_model.load_weights(str(checkpoint_path))

print(f"Model loaded from {checkpoint_path}")

## 6. Validation Inference
Run MC Dropout on validation set to fit calibration methods.

In [ ]:
# Validation MC Dropout
val_results = mc_model.predict_with_uncertainty(
 val_images,
 n_samples=config.n_mc_samples,
 verbose=True
)

val_predictions = val_results['mean']
val_variance = val_results['variance']

## 7. Calibration
Fit Temperature, Platt, and Isotonic calibration. Select method with lowest ECE.

In [ ]:
# Calibration

def compute_ece(probs, labels, n_bins=15):
 """Compute Expected Calibration Error."""
 bin_boundaries = np.linspace(0, 1, n_bins + 1)
 ece = 0.0
 for i in range(n_bins):
 in_bin = (probs > bin_boundaries[i]) & (probs <= bin_boundaries[i + 1])
 prop_in_bin = in_bin.mean()
 if prop_in_bin > 0:
 avg_conf = probs[in_bin].mean()
 avg_acc = labels[in_bin].mean()
 ece += np.abs(avg_acc - avg_conf) * prop_in_bin
 return ece

# Fit calibrators
temp_calibrator = TemperatureScaling()
temp_calibrator.fit(val_predictions, val_labels, verbose=False)
temp_ece = compute_ece(temp_calibrator.calibrate(val_predictions), val_labels)

platt_calibrator = PlattScaling()
platt_calibrator.fit(val_predictions, val_labels, verbose=False)
platt_ece = compute_ece(platt_calibrator.calibrate(val_predictions), val_labels)

iso_calibrator = IsotonicCalibration()
iso_calibrator.fit(val_predictions, val_labels, verbose=False)
iso_ece = compute_ece(iso_calibrator.calibrate(val_predictions), val_labels)

# Select best method
calibrators = {
 'temperature': (temp_calibrator, temp_ece),
 'platt': (platt_calibrator, platt_ece),
 'isotonic': (iso_calibrator, iso_ece)
}

best_method = min(calibrators, key=lambda x: calibrators[x][1])
best_calibrator = calibrators[best_method][0]

print(f"Calibration ECE: Temperature={temp_ece:.4f}, Platt={platt_ece:.4f}, Isotonic={iso_ece:.4f}")
print(f"Selected: {best_method}")

## 8. Test Inference
Run MC Dropout on test set to obtain predictions and uncertainty estimates.

In [ ]:
# Test MC Dropout
test_results = mc_model.predict_with_uncertainty(
 test_images,
 n_samples=config.n_mc_samples,
 verbose=True
)

predictions_uncal = test_results['mean']
variance = test_results['variance']
mc_samples = test_results['samples']

## 9. Apply Calibration
Apply selected calibration method to test predictions.

In [ ]:
# Apply Calibration
predictions = best_calibrator.calibrate(predictions_uncal)
binary_preds = (predictions > 0.5).astype(int)

## 10. Compute Metrics
Calculate AUC, accuracy, ECE, and variance ratio for uncertainty quality assessment.

In [ ]:
# Compute Metrics
ece = compute_ece(predictions, test_labels)
auc = roc_auc_score(test_labels, predictions)
accuracy = accuracy_score(test_labels, binary_preds)

correct_mask = binary_preds == test_labels
incorrect_mask = ~correct_mask
correct_variance = variance[correct_mask]
incorrect_variance = variance[incorrect_mask]
variance_ratio = incorrect_variance.mean() / correct_variance.mean()

if variance_ratio >= 1.5:
 quality = "Excellent"
elif variance_ratio >= 1.3:
 quality = "Good"
elif variance_ratio >= 1.1:
 quality = "Moderate"
else:
 quality = "Poor"

print("Performance Metrics")
print("-" * 40)
print(f"AUC-ROC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"ECE: {ece:.4f}")
print()
print("Uncertainty Metrics")
print("-" * 40)
print(f"Mean Variance: {variance.mean():.6f}")
print(f"Variance Ratio: {variance_ratio:.2f}x")
print(f"Quality: {quality}")

## 11. Reliability Diagram
Visualize calibration quality before and after calibration.

In [ ]:
# Reliability Diagram
results_path = BASE_PATH / 'results_mc_dropout'
results_path.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

def plot_reliability(ax, probs, labels, title):
 n_bins = 15
 bin_boundaries = np.linspace(0, 1, n_bins + 1)
 mean_predicted, fraction_positive = [], []
 
 for i in range(n_bins):
 in_bin = (probs > bin_boundaries[i]) & (probs <= bin_boundaries[i + 1])
 if in_bin.sum() > 0:
 mean_predicted.append(probs[in_bin].mean())
 fraction_positive.append(labels[in_bin].mean())
 
 ece_val = compute_ece(probs, labels)
 ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration', linewidth=1.5)
 ax.plot(mean_predicted, fraction_positive, 's-', markersize=6, 
 label=f'Model (ECE={ece_val:.3f})')
 ax.fill_between(mean_predicted, mean_predicted, fraction_positive, alpha=0.15)
 ax.set_xlabel('Predicted Probability')
 ax.set_ylabel('Observed Frequency')
 ax.set_title(title)
 ax.legend(loc='lower right')
 ax.set_xlim([0, 1])
 ax.set_ylim([0, 1])
 ax.set_aspect('equal')
 ax.grid(True, alpha=0.3)

plot_reliability(axes[0], predictions_uncal, test_labels, 'Before Calibration')
plot_reliability(axes[1], predictions, test_labels, f'After Calibration ({best_method.title()})')

plt.tight_layout()
plt.savefig(results_path / 'reliability_diagram.png', dpi=150)
plt.show()

## 12. Variance Distribution
Compare predictive variance between correct and incorrect predictions.

In [ ]:
# Variance Distribution
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(correct_variance, bins=50, alpha=0.7, label=f'Correct (n={correct_mask.sum()})',
 color='#2ecc71', density=True)
ax.hist(incorrect_variance, bins=50, alpha=0.7, label=f'Incorrect (n={incorrect_mask.sum()})',
 color='#e74c3c', density=True)
ax.axvline(correct_variance.mean(), color='#27ae60', linestyle='--', linewidth=2)
ax.axvline(incorrect_variance.mean(), color='#c0392b', linestyle='--', linewidth=2)
ax.set_xlabel('Predictive Variance')
ax.set_ylabel('Density')
ax.set_title(f'Variance Distribution by Prediction Correctness (Ratio: {variance_ratio:.2f}x)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_path / 'variance_distribution.png', dpi=150)
plt.show()

## 13. Clinical Routing
Route predictions by confidence level for clinical decision support.

In [ ]:
# Clinical Routing
calibrated_results = test_results.copy()
calibrated_results['mean'] = predictions

clinical_maker = ClinicalDecisionMaker(config)
decisions = clinical_maker.make_batch_decisions(mc_results=calibrated_results)

confidence_stats = {}
for level in ConfidenceLevel:
 mask = [d.confidence_level == level for d in decisions]
 count = sum(mask)
 if count > 0:
 indices = np.where(mask)[0]
 acc = (binary_preds[indices] == test_labels[indices]).mean()
 confidence_stats[level.value] = {'count': count, 'accuracy': acc}
 else:
 confidence_stats[level.value] = {'count': 0, 'accuracy': 0}

print("Clinical Decision Routing")
print("-" * 50)
print(f"{'Level':<12} {'Count':>8} {'Percent':>10} {'Accuracy':>10}")
print("-" * 50)
for level, stats in confidence_stats.items():
 pct = stats['count'] / len(decisions) * 100
 print(f"{level:<12} {stats['count']:>8} {pct:>9.1f}% {stats['accuracy']:>9.1%}")

auto_count = confidence_stats.get('high', {}).get('count', 0)
auto_acc = confidence_stats.get('high', {}).get('accuracy', 0)
print("-" * 50)
print(f"Auto-classify: {auto_count/len(decisions)*100:.1f}% at {auto_acc:.1%} accuracy")

## 14. Summary
Final results against target metrics.

In [ ]:
# Summary
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print()
print(f"{'Metric':<20} {'Value':>12} {'Target':>12} {'Status':>8}")
print("-" * 52)

ece_status = "PASS" if ece < 0.10 else "FAIL"
var_status = "PASS" if variance.mean() >= 0.01 else "FAIL"
ratio_status = "PASS" if variance_ratio >= 1.5 else "PARTIAL"

print(f"{'ECE':<20} {ece:>12.4f} {'< 0.10':>12} {ece_status:>8}")
print(f"{'Mean Variance':<20} {variance.mean():>12.6f} {'>= 0.01':>12} {var_status:>8}")
print(f"{'Variance Ratio':<20} {variance_ratio:>11.2f}x {'>= 1.5x':>12} {ratio_status:>8}")
print(f"{'AUC-ROC':<20} {auc:>12.4f} {'-':>12} {'-':>8}")
print(f"{'Accuracy':<20} {accuracy:>12.4f} {'-':>12} {'-':>8}")

## 15. Save Results
Export predictions, variances, and summary to disk.

In [ ]:
# Save Results
summary = {
 'config': {
 'mc_samples': config.n_mc_samples,
 'mc_dropout_rate': config.mc_dropout_rate,
 'spatial_dropout_rate': config.spatial_dropout_rate,
 'dense_units': config.dense_units,
 },
 'calibration': {
 'method': best_method,
 'ece': float(ece),
 },
 'performance': {
 'auc': float(auc),
 'accuracy': float(accuracy),
 },
 'uncertainty': {
 'mean_variance': float(variance.mean()),
 'median_variance': float(np.median(variance)),
 'variance_ratio': float(variance_ratio),
 'quality': quality,
 },
 'clinical': {
 'auto_classify_rate': float(auto_count / len(decisions)),
 'auto_classify_accuracy': float(auto_acc),
 },
 'targets': {
 'ece_met': bool(ece < 0.10),
 'variance_met': bool(variance.mean() >= 0.01),
 'ratio_met': bool(variance_ratio >= 1.5),
 }
}

with open(results_path / 'summary.json', 'w') as f:
 json.dump(summary, f, indent=2)

np.save(results_path / 'predictions.npy', predictions)
np.save(results_path / 'variances.npy', variance)
np.save(results_path / 'mc_samples.npy', mc_samples)

print(f"Results saved to {results_path}")